# EllipseNet Final Notebook

Core NumPy pipeline for YOLO-style ellipse detection on Pascal VOC.


In [ ]:
# Global setup: one import cell for the whole notebook.
import json
import os
import time
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
from PIL import Image

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/ellipsenet_matplotlib")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse as MplEllipse

# Make paths work from either the repo root or EllipseNet_Complete.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "EllipseNet_Final.ipynb").exists():
    candidate = PROJECT_DIR / "EllipseNet_Complete"
    if candidate.exists():
        PROJECT_DIR = candidate
os.chdir(PROJECT_DIR)

os.makedirs("figures", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
print("Notebook working directory:", PROJECT_DIR)


In [ ]:
# Pascal VOC class list and dataset defaults.
VOC_CLASSES = [
    "aeroplane", "bicycle", "bird", "boat", "bottle", "bus", "car", "cat",
    "chair", "cow", "diningtable", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "sofa", "train", "tvmonitor",
]

DEFAULT_SELECTED_CLASSES = tuple(VOC_CLASSES)
DEFAULT_ROTATION_ANGLES = (0.0, -20.0, -10.0, 10.0, 20.0)
DEFAULT_RANDOM_ROTATION_RANGE = (-20.0, 20.0)
DEFAULT_VOC_SEARCH_ROOT = "/Users/haile/Downloads/archive"


def bbox_to_ellipse(xmin, ymin, xmax, ymax, img_w, img_h):
    """Convert a VOC box to a normalized axis-aligned ellipse."""
    cx = ((xmin + xmax) * 0.5) / img_w
    cy = ((ymin + ymax) * 0.5) / img_h
    a = ((xmax - xmin) * 0.5) / img_w
    b = ((ymax - ymin) * 0.5) / img_h
    return cx, cy, a, b, 0.0

def normalise_theta(theta):
    """Normalize ellipse orientation to [0, pi)."""
    return float(theta % np.pi)

def find_voc_root(search_root=DEFAULT_VOC_SEARCH_ROOT):
    """
    Locate a VOC directory containing JPEGImages, Annotations, and ImageSets.

    The Kaggle archive used for this project has nested folders, so callers can
    pass either /Users/haile/Downloads/archive or the final VOC2012 directory.
    """
    required = ("JPEGImages", "Annotations", "ImageSets")
    if all(os.path.isdir(os.path.join(search_root, name)) for name in required):
        return search_root

    for root, dirs, _ in os.walk(search_root):
        if all(name in dirs for name in required):
            return root
    raise FileNotFoundError(
        f"Could not find a Pascal VOC root under {search_root!r}. "
        "Expected JPEGImages/, Annotations/, and ImageSets/."
    )

def _read_split_ids(voc_root, split):
    split_path = os.path.join(voc_root, "ImageSets", "Main", f"{split}.txt")
    if not os.path.exists(split_path):
        raise FileNotFoundError(f"VOC split file not found: {split_path}")
    with open(split_path) as f:
        return [line.strip().split()[0] for line in f if line.strip()]

def _parse_voc_xml(xml_path, selected_classes, include_difficult=False):
    class_to_idx = {name: idx for idx, name in enumerate(selected_classes)}
    root = ET.parse(xml_path).getroot()
    image_id = os.path.splitext(os.path.basename(xml_path))[0]
    filename = root.findtext("filename", f"{image_id}.jpg")
    size = root.find("size")
    img_w = int(size.findtext("width"))
    img_h = int(size.findtext("height"))

    targets = []
    for obj in root.iter("object"):
        name = obj.findtext("name", "")
        if name not in class_to_idx:
            continue
        difficult = int(obj.findtext("difficult", "0"))
        if difficult and not include_difficult:
            continue

        box = obj.find("bndbox")
        xmin = float(box.findtext("xmin"))
        ymin = float(box.findtext("ymin"))
        xmax = float(box.findtext("xmax"))
        ymax = float(box.findtext("ymax"))
        xmin = max(0.0, min(xmin, img_w - 1.0))
        xmax = max(0.0, min(xmax, img_w - 1.0))
        ymin = max(0.0, min(ymin, img_h - 1.0))
        ymax = max(0.0, min(ymax, img_h - 1.0))
        if xmax <= xmin or ymax <= ymin:
            continue

        cx, cy, a, b, theta = bbox_to_ellipse(xmin, ymin, xmax, ymax, img_w, img_h)
        targets.append({
            "class_id": class_to_idx[name],
            "class_name": name,
            "cx": round(cx, 6),
            "cy": round(cy, 6),
            "a": round(a, 6),
            "b": round(b, 6),
            "theta": round(theta, 6),
        })

    return {
        "image_id": image_id,
        "file_name": filename,
        "img_w": img_w,
        "img_h": img_h,
        "targets": targets,
    }

def load_voc_samples(voc_root=None, split="train", selected_classes=None,
                     max_images=None, include_difficult=False):
    """
    Load VOC samples with at least one selected object.

    Returns a list of dicts with image metadata and ellipse targets.
    """
    voc_root = find_voc_root(voc_root or DEFAULT_VOC_SEARCH_ROOT)
    selected_classes = tuple(selected_classes or DEFAULT_SELECTED_CLASSES)
    image_ids = _read_split_ids(voc_root, split) if split else [
        os.path.splitext(name)[0]
        for name in sorted(os.listdir(os.path.join(voc_root, "Annotations")))
        if name.endswith(".xml")
    ]

    samples = []
    for image_id in image_ids:
        xml_path = os.path.join(voc_root, "Annotations", f"{image_id}.xml")
        if not os.path.exists(xml_path):
            continue
        sample = _parse_voc_xml(xml_path, selected_classes, include_difficult)
        if sample["targets"]:
            samples.append(sample)
            if max_images is not None and len(samples) >= max_images:
                break
    return samples


In [ ]:
# Dataset class: resize first, then apply one random rotation per sample access.
def rotate_normalized_point(cx, cy, angle_deg):
    """Rotate a normalized point around image center in theta convention."""
    angle = np.deg2rad(angle_deg)
    dx = cx - 0.5
    dy = cy - 0.5
    cos_t = np.cos(angle)
    sin_t = np.sin(angle)
    x_new = 0.5 + cos_t * dx - sin_t * dy
    y_new = 0.5 + sin_t * dx + cos_t * dy
    return float(x_new), float(y_new)

class VOCEllipseDataset:
    """
    Pascal VOC dataset returning (image, targets) for EllipseNet.

    Images are resized to img_size x img_size before any rotation. Passing
    rotation_angles="random" applies one random rotation per loaded sample
    instead of creating multiple virtual variants.
    """

    def __init__(self, voc_root=None, split="train", selected_classes=None,
                 img_size=64, rotation_angles=None, max_images=None,
                 include_difficult=False):
        self.voc_root = find_voc_root(voc_root or DEFAULT_VOC_SEARCH_ROOT)
        self.img_dir = os.path.join(self.voc_root, "JPEGImages")
        self.selected_classes = tuple(selected_classes or DEFAULT_SELECTED_CLASSES)
        self.img_size = int(img_size)
        self.random_rotation = rotation_angles == "random"
        self.rotation_angles = (0.0,) if self.random_rotation else tuple(
            DEFAULT_ROTATION_ANGLES if rotation_angles is None else rotation_angles
        )
        self.samples = load_voc_samples(
            self.voc_root,
            split=split,
            selected_classes=self.selected_classes,
            max_images=max_images,
            include_difficult=include_difficult,
        )
        if not self.samples:
            raise ValueError("No VOC samples matched the requested split/classes.")

    def __len__(self):
        return len(self.samples) if self.random_rotation else len(self.samples) * len(self.rotation_angles)

    def __getitem__(self, idx):
        if self.random_rotation:
            sample_idx = idx
            angle_deg = float(np.random.uniform(*DEFAULT_RANDOM_ROTATION_RANGE))
        else:
            sample_idx = idx // len(self.rotation_angles)
            angle_deg = self.rotation_angles[idx % len(self.rotation_angles)]
        sample = self.samples[sample_idx]

        img_path = os.path.join(self.img_dir, sample["file_name"])
        img = Image.open(img_path).convert("RGB")
        # Resize first so each sample is rotated only once at training resolution.
        img = img.resize((self.img_size, self.img_size), Image.BILINEAR)

        targets = [dict(t) for t in sample["targets"]]
        if abs(angle_deg) > 1e-9:
            img, targets = self._rotate_sample(img, targets, angle_deg)

        img_arr = np.array(img, dtype=np.float32) / 255.0
        return img_arr.transpose(2, 0, 1), targets

    def sample_info(self, idx):
        if self.random_rotation:
            sample_idx = idx
            angle_deg = "random"
        else:
            sample_idx = idx // len(self.rotation_angles)
            angle_deg = self.rotation_angles[idx % len(self.rotation_angles)]
        sample = self.samples[sample_idx]
        return {
            "image_id": sample["image_id"],
            "file_name": sample["file_name"],
            "rotation_angle_deg": angle_deg,
        }

    def _rotate_sample(self, img, targets, angle_deg):
        # Ellipse theta uses image coordinates where positive angles rotate
        # clockwise; PIL's positive rotate is counter-clockwise.
        rotated_img = img.rotate(
            -angle_deg,
            resample=Image.BILINEAR,
            expand=False,
            fillcolor=(128, 128, 128),
        )
        theta_delta = np.deg2rad(angle_deg)
        rotated_targets = []
        for target in targets:
            cx, cy = rotate_normalized_point(target["cx"], target["cy"], angle_deg)
            if not (0.0 <= cx <= 1.0 and 0.0 <= cy <= 1.0):
                continue
            new_target = dict(target)
            new_target["cx"] = float(round(cx, 6))
            new_target["cy"] = float(round(cy, 6))
            new_target["theta"] = float(round(normalise_theta(target["theta"] + theta_delta), 6))
            rotated_targets.append(new_target)
        return rotated_img, rotated_targets


In [ ]:
# Training dataset configuration.
CLASS_NAMES = list(DEFAULT_SELECTED_CLASSES)
C = len(CLASS_NAMES)
B = 2
IMG_SIZE = 448
S = 7
MAX_SAMPLES = 500

VOC_ROOT = find_voc_root(os.environ.get("VOC_ROOT", "../archive/VOC2012_train_val/VOC2012_train_val"))

train_ds = VOCEllipseDataset(
    voc_root=VOC_ROOT,
    split="train",
    selected_classes=CLASS_NAMES,
    img_size=IMG_SIZE,
    rotation_angles="random",
    max_images=MAX_SAMPLES,
)
val_ds = VOCEllipseDataset(
    voc_root=VOC_ROOT,
    split="val",
    selected_classes=CLASS_NAMES,
    img_size=IMG_SIZE,
    rotation_angles=(0.0,),
    max_images=MAX_SAMPLES,
)

print("VOC root:", VOC_ROOT)
print("Classes:", len(CLASS_NAMES), CLASS_NAMES)
print("Image size:", IMG_SIZE)
print("Grid size:", S)
print("Anchors per cell:", B)
print("Train samples:", len(train_ds))
print("Validation samples:", len(val_ds))
print("Example target:", train_ds[0][1][0])


In [ ]:
# Quick visual check for resized images and ground-truth ellipses.
def show_dataset_samples(dataset, class_names, n_images=6, out_path="figures/sample_ground_truth.png"):
    n_images = min(n_images, len(dataset))
    cols = 3
    rows = int(np.ceil(n_images / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for ax, idx in zip(axes, range(n_images)):
        img, targets = dataset[idx]
        display_img = np.clip(img.transpose(1, 2, 0), 0, 1)
        ax.imshow(display_img)
        h, w = display_img.shape[:2]
        for target in targets:
            ellipse = MplEllipse(
                (target["cx"] * w, target["cy"] * h),
                2 * target["a"] * w,
                2 * target["b"] * h,
                angle=np.rad2deg(target["theta"]),
                fill=False,
                edgecolor="lime",
                linewidth=1.5,
            )
            ax.add_patch(ellipse)
            ax.text(target["cx"] * w, target["cy"] * h, class_names[target["class_id"]], color="white", fontsize=8)

    fig.tight_layout()
    fig.savefig(out_path, dpi=140)
    plt.show()
    return out_path

show_dataset_samples(train_ds, CLASS_NAMES)


In [ ]:
# im2col helpers used by the NumPy convolution layers.
def im2col(x, kH, kW, stride=1, pad=0):
    """Convert image patches to columns for efficient convolution."""
    N, C, H, W = x.shape
    x_pad = np.pad(x, ((0,0),(0,0),(pad,pad),(pad,pad)), mode='constant')
    out_H = (H + 2*pad - kH) // stride + 1
    out_W = (W + 2*pad - kW) // stride + 1
    col = np.zeros((N, C, kH, kW, out_H, out_W))
    for r in range(kH):
        r_max = r + stride * out_H
        for c in range(kW):
            c_max = c + stride * out_W
            col[:, :, r, c, :, :] = x_pad[:, :, r:r_max:stride, c:c_max:stride]
    col = col.transpose(0, 4, 5, 1, 2, 3).reshape(N * out_H * out_W, -1)
    return col, out_H, out_W

def col2im(col, x_shape, kH, kW, stride=1, pad=0):
    """Inverse of im2col — accumulate gradients back to image space."""
    N, C, H, W = x_shape
    out_H = (H + 2*pad - kH) // stride + 1
    out_W = (W + 2*pad - kW) // stride + 1
    col = col.reshape(N, out_H, out_W, C, kH, kW).transpose(0, 3, 4, 5, 1, 2)
    x_pad = np.zeros((N, C, H + 2*pad, W + 2*pad))
    for r in range(kH):
        r_max = r + stride * out_H
        for c in range(kW):
            c_max = c + stride * out_W
            x_pad[:, :, r:r_max:stride, c:c_max:stride] += col[:, :, r, c, :, :]
    if pad == 0:
        return x_pad
    return x_pad[:, :, pad:-pad, pad:-pad]


In [ ]:
# Core NumPy layers with manual backward passes.
class Conv2D:
    """
    2-D cross-correlation layer.
    W shape: (out_channels, in_channels, kH, kW)
    """
    def __init__(self, in_ch, out_ch, kernel=3, stride=1, pad=None):
        self.in_ch = in_ch
        self.out_ch = out_ch
        self.k = kernel
        self.stride = stride
        self.pad = kernel // 2 if pad is None else pad
        # He initialisation
        std = np.sqrt(2.0 / (in_ch * kernel * kernel))
        self.W = np.random.randn(out_ch, in_ch, kernel, kernel) * std
        self.b = np.zeros(out_ch)
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)
        self._cache = None

    def forward(self, x, training=True):
        N, C, H, W_img = x.shape
        col, oH, oW = im2col(x, self.k, self.k, self.stride, self.pad)
        W_flat = self.W.reshape(self.out_ch, -1)          # (out_ch, C*k*k)
        out = col @ W_flat.T + self.b                      # (N*oH*oW, out_ch)
        out = out.reshape(N, oH, oW, self.out_ch).transpose(0, 3, 1, 2)
        if training:
            self._cache = (x, col, oH, oW)
        return out

    def backward(self, grad_out):
        x, col, oH, oW = self._cache
        N = x.shape[0]
        # grad_out: (N, out_ch, oH, oW)
        g = grad_out.transpose(0, 2, 3, 1).reshape(-1, self.out_ch)  # (N*oH*oW, out_ch)
        self.dW = (g.T @ col).reshape(self.W.shape)
        self.db = g.sum(axis=0)
        W_flat = self.W.reshape(self.out_ch, -1)
        d_col = g @ W_flat                                 # (N*oH*oW, C*k*k)
        grad_in = col2im(d_col, x.shape, self.k, self.k, self.stride, self.pad)
        return grad_in

    def params(self):
        return [(self.W, self.dW), (self.b, self.db)]

class Conv1x1(Conv2D):
    """Pointwise convolution — inherits Conv2D with kernel=1, pad=0."""
    def __init__(self, in_ch, out_ch):
        super().__init__(in_ch, out_ch, kernel=1, stride=1, pad=0)

class MaxPool2D:
    def __init__(self, pool=2, stride=2):
        self.pool = pool
        self.stride = stride
        self._cache = None

    def forward(self, x, training=True):
        N, C, H, W = x.shape
        p, s = self.pool, self.stride
        oH = (H - p) // s + 1
        oW = (W - p) // s + 1
        out = np.zeros((N, C, oH, oW))
        mask = np.zeros_like(x, dtype=bool)
        for i in range(oH):
            for j in range(oW):
                patch = x[:, :, i*s:i*s+p, j*s:j*s+p]         # (N,C,p,p)
                mx = patch.max(axis=(2,3), keepdims=True)
                m = (patch == mx)
                # break ties: keep first occurrence
                flat = m.reshape(N, C, -1)
                first = np.zeros_like(flat)
                idx = flat.argmax(axis=2)
                first[np.arange(N)[:,None], np.arange(C)[None,:], idx] = True
                m = first.reshape(N, C, p, p)
                mask[:, :, i*s:i*s+p, j*s:j*s+p] |= m
                out[:, :, i, j] = (patch * m).sum(axis=(2,3))
        if training:
            self._cache = (x.shape, mask, oH, oW)
        return out

    def backward(self, grad_out):
        x_shape, mask, oH, oW = self._cache
        N, C, H, W = x_shape
        p, s = self.pool, self.stride
        grad_in = np.zeros(x_shape)
        for i in range(oH):
            for j in range(oW):
                g = grad_out[:, :, i, j][:, :, None, None]  # broadcast
                grad_in[:, :, i*s:i*s+p, j*s:j*s+p] += g * mask[:, :, i*s:i*s+p, j*s:j*s+p]
        return grad_in

    def params(self):
        return []

class ReLU:
    def __init__(self):
        self._cache = None

    def forward(self, x, training=True):
        if training:
            self._cache = x
        return np.maximum(0, x)

    def backward(self, grad_out):
        return grad_out * (self._cache > 0)

    def params(self):
        return []

class LeakyReLU:
    def __init__(self, alpha=0.1):
        self.alpha = alpha
        self._cache = None

    def forward(self, x, training=True):
        if training:
            self._cache = x
        return np.where(x > 0, x, self.alpha * x)

    def backward(self, grad_out):
        return grad_out * np.where(self._cache > 0, 1.0, self.alpha)

    def params(self):
        return []

class BatchNorm2D:
    """
    Batch normalisation over (N, H, W) per channel.
    Running stats used at inference time.
    """
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        self.eps = eps
        self.momentum = momentum
        self.gamma = np.ones(num_features)
        self.beta = np.zeros(num_features)
        self.dgamma = np.zeros_like(self.gamma)
        self.dbeta = np.zeros_like(self.beta)
        self.running_mean = np.zeros(num_features)
        self.running_var = np.ones(num_features)
        self._cache = None

    def forward(self, x, training=True):
        # x: (N, C, H, W)
        N, C, H, W = x.shape
        if training:
            mean = x.mean(axis=(0, 2, 3))           # (C,)
            var  = x.var(axis=(0, 2, 3))
            x_norm = (x - mean[None,:,None,None]) / np.sqrt(var[None,:,None,None] + self.eps)
            self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * mean
            self.running_var  = (1 - self.momentum) * self.running_var  + self.momentum * var
            self._cache = (x, x_norm, mean, var)
        else:
            x_norm = (x - self.running_mean[None,:,None,None]) / np.sqrt(self.running_var[None,:,None,None] + self.eps)
        return self.gamma[None,:,None,None] * x_norm + self.beta[None,:,None,None]

    def backward(self, grad_out):
        x, x_norm, mean, var = self._cache
        N, C, H, W = x.shape
        m = N * H * W
        self.dgamma = (grad_out * x_norm).sum(axis=(0,2,3))
        self.dbeta  = grad_out.sum(axis=(0,2,3))
        dx_norm = grad_out * self.gamma[None,:,None,None]
        dvar = (dx_norm * (x - mean[None,:,None,None]) * -0.5 *
                (var[None,:,None,None] + self.eps)**(-1.5)).sum(axis=(0,2,3))
        dmean = (dx_norm * -1 / np.sqrt(var[None,:,None,None] + self.eps)).sum(axis=(0,2,3)) \
                + dvar * (-2 * (x - mean[None,:,None,None])).mean(axis=(0,2,3))
        grad_in = (dx_norm / np.sqrt(var[None,:,None,None] + self.eps)
                   + dvar[None,:,None,None] * 2 * (x - mean[None,:,None,None]) / m
                   + dmean[None,:,None,None] / m)
        return grad_in

    def params(self):
        return [(self.gamma, self.dgamma), (self.beta, self.dbeta)]

class Dense:
    def __init__(self, in_dim, out_dim):
        std = np.sqrt(2.0 / in_dim)
        self.W = np.random.randn(in_dim, out_dim) * std
        self.b = np.zeros(out_dim)
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)
        self._cache = None

    def forward(self, x, training=True):
        if training:
            self._cache = x
        return x @ self.W + self.b

    def backward(self, grad_out):
        x = self._cache
        self.dW = x.T @ grad_out
        self.db = grad_out.sum(axis=0)
        return grad_out @ self.W.T

    def params(self):
        return [(self.W, self.dW), (self.b, self.db)]

class Dropout:
    def __init__(self, rate=0.5):
        self.rate = rate
        self._mask = None

    def forward(self, x, training=True):
        if training:
            self._mask = (np.random.rand(*x.shape) > self.rate) / (1 - self.rate)
            return x * self._mask
        return x

    def backward(self, grad_out):
        return grad_out * self._mask

    def params(self):
        return []


In [ ]:
# Optimizer and learning-rate schedule.
class Adam:
    """
    Adam optimizer — Kingma & Ba 2015.
    Maintains per-parameter first and second moment estimates.
    """
    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.0):
        """
        params : list of (param_array, grad_array) tuples
        """
        self.params = params
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.weight_decay = weight_decay
        self.t = 0
        self.m = [np.zeros_like(p) for p, _ in params]
        self.v = [np.zeros_like(p) for p, _ in params]

    def step(self):
        self.t += 1
        for i, (p, g) in enumerate(self.params):
            if self.weight_decay > 0:
                g = g + self.weight_decay * p
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * g
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * g**2
            m_hat = self.m[i] / (1 - self.beta1**self.t)
            v_hat = self.v[i] / (1 - self.beta2**self.t)
            p -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

    def zero_grad(self):
        for _, g in self.params:
            g[:] = 0.0

    def set_lr(self, lr):
        self.lr = lr

def warmup_cosine_lr(step, warmup_steps, total_steps, base_lr, min_lr=1e-6):
    """Warmup then cosine decay."""
    if step < warmup_steps:
        return base_lr * step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return min_lr + 0.5 * (base_lr - min_lr) * (1 + np.cos(np.pi * progress))


In [ ]:
# Ellipse geometry, activations, and classification helpers.
def point_in_ellipse(px, py, cx, cy, a, b, theta):
    """
    Returns True if point (px, py) lies inside ellipse (cx,cy,a,b,theta).
    Works on numpy arrays; px, py may be arrays.
    Ellipse equation in rotated frame:
        ((x')/a)² + ((y')/b)² ≤ 1
    where x' = (px-cx)cos(θ) + (py-cy)sin(θ)
          y' =-(px-cx)sin(θ) + (py-cy)cos(θ)
    """
    a   = max(a, 1e-6)
    b   = max(b, 1e-6)
    cos_t = np.cos(theta)
    sin_t = np.sin(theta)
    dx = px - cx
    dy = py - cy
    xr = dx * cos_t + dy * sin_t
    yr = -dx * sin_t + dy * cos_t
    return (xr**2 / a**2 + yr**2 / b**2) <= 1.0

def ellipse_iou_mc(e1, e2, n_samples=500):
    """
    Monte Carlo estimate of Intersection-over-Union between two ellipses.

    e1, e2 : (cx, cy, a, b, theta) — normalised coords [0,1]
    Returns scalar IoU in [0, 1].
    """
    cx1, cy1, a1, b1, t1 = e1
    cx2, cy2, a2, b2, t2 = e2

    a1, b1 = max(a1, 1e-6), max(b1, 1e-6)
    a2, b2 = max(a2, 1e-6), max(b2, 1e-6)

    # Axis-aligned bounding box of each ellipse
    def ellipse_aabb(cx, cy, a, b, theta):
        tx = np.sqrt(a**2 * np.cos(theta)**2 + b**2 * np.sin(theta)**2)
        ty = np.sqrt(a**2 * np.sin(theta)**2 + b**2 * np.cos(theta)**2)
        return cx - tx, cy - ty, cx + tx, cy + ty

    x1min, y1min, x1max, y1max = ellipse_aabb(cx1, cy1, a1, b1, t1)
    x2min, y2min, x2max, y2max = ellipse_aabb(cx2, cy2, a2, b2, t2)

    ux_min = min(x1min, x2min)
    uy_min = min(y1min, y2min)
    ux_max = max(x1max, x2max)
    uy_max = max(y1max, y2max)

    if ux_max <= ux_min or uy_max <= uy_min:
        return 0.0

    px = np.random.uniform(ux_min, ux_max, n_samples)
    py = np.random.uniform(uy_min, uy_max, n_samples)

    in1 = point_in_ellipse(px, py, cx1, cy1, a1, b1, t1)
    in2 = point_in_ellipse(px, py, cx2, cy2, a2, b2, t2)

    intersection = np.logical_and(in1, in2).sum()
    union        = np.logical_or (in1, in2).sum()

    if union == 0:
        return 0.0
    return float(intersection) / float(union)

def batch_ellipse_iou(pred_ellipses, gt_ellipses, n_samples=300):
    """
    Compute IoU for matched pairs.
    pred_ellipses, gt_ellipses : (N, 5) arrays
    Returns (N,) IoU values.
    """
    N = pred_ellipses.shape[0]
    ious = np.zeros(N)
    for i in range(N):
        ious[i] = ellipse_iou_mc(pred_ellipses[i], gt_ellipses[i], n_samples)
    return ious

def angle_loss(theta_pred, theta_gt):
    """
    Smooth angle loss robust to π-periodicity:
        L = min(|p - g|, π - |p - g|)
    Works on arrays.
    """
    diff = np.abs(theta_pred - theta_gt)
    return np.minimum(diff, np.pi - diff)

def bce(pred, target, eps=1e-7):
    """Element-wise BCE. pred in (0,1)."""
    pred = np.clip(pred, eps, 1 - eps)
    return -(target * np.log(pred) + (1 - target) * np.log(1 - pred))

def bce_grad(pred, target, eps=1e-7):
    """Gradient of BCE w.r.t. pred."""
    pred = np.clip(pred, eps, 1 - eps)
    return -(target / pred) + (1 - target) / (1 - pred)

def softmax(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

def cross_entropy(logits, labels, eps=1e-7):
    """logits: (N, C), labels: (N,) int.  Returns scalar."""
    probs = softmax(logits)
    N = logits.shape[0]
    if N == 0:
        return 0.0
    return -np.log(probs[np.arange(N), labels] + eps).mean()

def cross_entropy_grad(logits, labels):
    """Gradient of softmax-CE w.r.t. logits."""
    probs = softmax(logits)
    N = logits.shape[0]
    if N == 0:
        return np.zeros_like(logits)
    probs[np.arange(N), labels] -= 1
    return probs / N

def sigmoid(x):
    return np.where(x >= 0,
                    1 / (1 + np.exp(-x)),
                    np.exp(x) / (1 + np.exp(x)))

def sigmoid_grad(x):
    s = sigmoid(x)
    return s * (1 - s)


In [ ]:
# YOLO-style ellipse loss: localization, objectness, class, and theta terms.
class EllipseLoss:
    """
    Computes the combined detection loss and returns (scalar_loss, grad_dict).

    Output tensor expected shape: (N, S, S, B, 6 + C)
    where dim-5 layout is:
        [0]      raw objectness logit
        [1:3]    raw cx, cy offsets (pre-sigmoid)
        [3:5]    raw log_a, log_b offsets
        [5]      raw theta (pre-pi*sigmoid)
        [6:]     raw class logits

    targets : list (len=N) of lists of dicts with keys
        {class_id, cx, cy, a, b, theta}
    """

    def __init__(self, S=8, B=2, C=3, anchors=None,
                 lambda_loc=5.0, lambda_obj=1.0, lambda_noobj=0.5,
                 lambda_cls=1.0, lambda_theta=2.0):
        self.S = S
        self.B = B
        self.C = C
        self.lambda_loc   = lambda_loc
        self.lambda_obj   = lambda_obj
        self.lambda_noobj = lambda_noobj
        self.lambda_cls   = lambda_cls
        self.lambda_theta = lambda_theta

        if anchors is None:
            # default anchor semi-axes (normalised), shape (B, 2)
            self.anchors = np.array([
                [0.10, 0.07],
                [0.20, 0.14],
            ])
        else:
            self.anchors = np.array(anchors)

    # ─── decode raw output → (cx, cy, a, b, theta, conf, cls_probs) ───
    def decode(self, raw, training=True):
        """
        raw : (N, S, S, B, 6+C)
        Returns decoded dict with same leading dims.
        """
        S = self.S
        conf  = sigmoid(raw[..., 0])         # objectness
        # cx, cy: offset from grid cell, in [0,1] within cell
        cx_off = sigmoid(raw[..., 1])
        cy_off = sigmoid(raw[..., 2])
        # absolute centre (normalised to full image)
        grid_x = np.arange(S).reshape(1, 1, S, 1)
        grid_y = np.arange(S).reshape(1, S, 1, 1)
        cx = (cx_off + grid_x) / S
        cy = (cy_off + grid_y) / S
        # semi-axes: exp(raw) * anchor
        a = np.exp(raw[..., 3]) * self.anchors[None, None, None, :, 0]
        b = np.exp(raw[..., 4]) * self.anchors[None, None, None, :, 1]
        # theta in [0, pi)
        theta = np.pi * sigmoid(raw[..., 5])
        cls_logits = raw[..., 6:]
        return conf, cx, cy, a, b, theta, cls_logits

    # ─── build target tensors ─────────────────────────────────────────
    def _build_targets(self, targets, N):
        """
        Assign each ground-truth ellipse to the best-matching anchor/cell.
        Returns boolean masks and target arrays (all numpy).
        """
        S, B, C = self.S, self.B, self.C
        obj_mask   = np.zeros((N, S, S, B), dtype=bool)
        noobj_mask = np.ones ((N, S, S, B), dtype=bool)
        tgt_cx    = np.zeros((N, S, S, B))
        tgt_cy    = np.zeros((N, S, S, B))
        tgt_a     = np.zeros((N, S, S, B))
        tgt_b     = np.zeros((N, S, S, B))
        tgt_theta = np.zeros((N, S, S, B))
        tgt_cls   = np.zeros((N, S, S, B), dtype=int)

        for ni, img_targets in enumerate(targets):
            for gt in img_targets:
                cx, cy = gt['cx'], gt['cy']
                a,  b  = gt['a'],  gt['b']
                theta  = gt['theta']
                cls    = int(gt['class_id'])

                # which grid cell
                gi = min(int(cx * S), S - 1)
                gj = min(int(cy * S), S - 1)

                # which anchor best matches by aspect-ratio IoU (axis-aligned)
                best_iou = -1
                best_b   = 0
                for bi in range(B):
                    an_a, an_b = self.anchors[bi]
                    # Jaccard between two axis-aligned ellipses centred at origin
                    inter_a = min(a, an_a)
                    inter_b = min(b, an_b)
                    inter   = np.pi * inter_a * inter_b
                    union   = np.pi * (a * b + an_a * an_b - inter_a * inter_b)
                    iou     = inter / max(union, 1e-9)
                    if iou > best_iou:
                        best_iou = iou
                        best_b   = bi

                obj_mask  [ni, gj, gi, best_b] = True
                noobj_mask[ni, gj, gi, best_b] = False
                # targets are offsets / log-scale
                tgt_cx   [ni, gj, gi, best_b] = cx * S - gi
                tgt_cy   [ni, gj, gi, best_b] = cy * S - gj
                tgt_a    [ni, gj, gi, best_b] = np.log(max(a  / self.anchors[best_b, 0], 1e-6))
                tgt_b    [ni, gj, gi, best_b] = np.log(max(b  / self.anchors[best_b, 1], 1e-6))
                tgt_theta[ni, gj, gi, best_b] = theta
                tgt_cls  [ni, gj, gi, best_b] = cls

        return (obj_mask, noobj_mask,
                tgt_cx, tgt_cy, tgt_a, tgt_b, tgt_theta, tgt_cls)

    # ─── main loss compute ────────────────────────────────────────────
    def compute(self, raw, targets):
        """
        raw     : (N, S, S, B, 6+C) — raw network outputs
        targets : list of N lists of gt dicts
        Returns (loss_scalar, grad_raw)
        """
        N, S, _, B, outd = raw.shape
        expected_outd = 6 + self.C
        if S != self.S or raw.shape[2] != self.S or B != self.B or outd != expected_outd:
            raise ValueError(
                f"Loss expected raw shape (N,{self.S},{self.S},{self.B},{expected_outd}), "
                f"got {raw.shape}."
            )
        grad_raw = np.zeros_like(raw)

        (obj_mask, noobj_mask,
         tgt_cx, tgt_cy, tgt_a, tgt_b, tgt_theta, tgt_cls) = self._build_targets(targets, N)

        # ── decode predictions ────────────────────────────────────────
        conf_raw  = raw[..., 0]
        cx_raw    = raw[..., 1]
        cy_raw    = raw[..., 2]
        a_raw     = raw[..., 3]
        b_raw     = raw[..., 4]
        theta_raw = raw[..., 5]

        conf      = sigmoid(conf_raw)
        cx_pred   = sigmoid(cx_raw)
        cy_pred   = sigmoid(cy_raw)
        theta_pred = np.pi * sigmoid(theta_raw)

        # ── objectness loss ───────────────────────────────────────────
        loss_obj   = bce(conf[obj_mask],   np.ones (obj_mask.sum())).mean()   if obj_mask.any()   else 0.0
        loss_noobj = bce(conf[noobj_mask], np.zeros(noobj_mask.sum())).mean() if noobj_mask.any() else 0.0

        g_conf_obj = np.zeros_like(conf)
        g_conf_noobj = np.zeros_like(conf)
        if obj_mask.any():
            g_conf_obj[obj_mask] = bce_grad(conf[obj_mask], np.ones(obj_mask.sum())) / max(obj_mask.sum(), 1)
        if noobj_mask.any():
            g_conf_noobj[noobj_mask] = bce_grad(conf[noobj_mask], np.zeros(noobj_mask.sum())) / max(noobj_mask.sum(), 1)
        g_conf = self.lambda_obj * g_conf_obj + self.lambda_noobj * g_conf_noobj
        grad_raw[..., 0] = g_conf * sigmoid_grad(conf_raw)

        n_obj = obj_mask.sum()
        if n_obj == 0:
            total_loss = (self.lambda_obj * loss_obj + self.lambda_noobj * loss_noobj)
            loss_dict = dict(
                total=total_loss, loc=0.0,
                obj=loss_obj, noobj=loss_noobj,
                cls=0.0, theta=0.0
            )
            return total_loss, grad_raw, loss_dict

        # ── coordinate losses (responsible anchors only) ───────────────
        # cx, cy: MSE(sigmoid(pred), target)
        cx_err = cx_pred[obj_mask] - tgt_cx[obj_mask]
        cy_err = cy_pred[obj_mask] - tgt_cy[obj_mask]
        loss_cx = (cx_err**2).mean()
        loss_cy = (cy_err**2).mean()

        g_cx_raw = np.zeros_like(cx_raw)
        g_cy_raw = np.zeros_like(cy_raw)
        g_cx_raw[obj_mask] = 2 * cx_err / n_obj * sigmoid_grad(cx_raw[obj_mask])
        g_cy_raw[obj_mask] = 2 * cy_err / n_obj * sigmoid_grad(cy_raw[obj_mask])
        grad_raw[..., 1] = g_cx_raw
        grad_raw[..., 2] = g_cy_raw

        # a, b: MSE in log-space
        a_err = a_raw[obj_mask] - tgt_a[obj_mask]
        b_err = b_raw[obj_mask] - tgt_b[obj_mask]
        loss_ab = ((a_err**2).mean() + (b_err**2).mean()) * 0.5

        g_a = np.zeros_like(a_raw); g_a[obj_mask] = 2 * a_err / n_obj
        g_b = np.zeros_like(b_raw); g_b[obj_mask] = 2 * b_err / n_obj
        grad_raw[..., 3] = g_a
        grad_raw[..., 4] = g_b

        loss_loc = loss_cx + loss_cy + loss_ab

        # ── theta loss ────────────────────────────────────────────────
        al = angle_loss(theta_pred[obj_mask], tgt_theta[obj_mask])
        loss_theta = al.mean()
        # gradient of min(|diff|, π-|diff|) w.r.t. theta_pred
        diff = theta_pred[obj_mask] - tgt_theta[obj_mask]
        sign = np.sign(diff)
        use_direct = np.abs(diff) <= (np.pi - np.abs(diff))
        dangle_dtheta_pred = np.where(use_direct, sign, -sign)
        g_theta_raw = np.zeros_like(theta_raw)
        g_theta_raw[obj_mask] = (dangle_dtheta_pred / n_obj) * np.pi * sigmoid_grad(theta_raw[obj_mask])
        grad_raw[..., 5] = g_theta_raw

        # ── classification loss ───────────────────────────────────────
        cls_logits = raw[..., 6:]                         # (N,S,S,B,C)
        cls_pred   = cls_logits[obj_mask]                 # (n_obj, C)
        cls_target = tgt_cls[obj_mask]                    # (n_obj,)
        loss_cls   = cross_entropy(cls_pred, cls_target)
        g_cls      = np.zeros_like(cls_logits)
        g_cls[obj_mask] = cross_entropy_grad(cls_pred, cls_target)
        grad_raw[..., 6:] = g_cls

        # ── total loss ────────────────────────────────────────────────
        total_loss = (self.lambda_loc   * loss_loc
                    + self.lambda_obj   * loss_obj
                    + self.lambda_noobj * loss_noobj
                    + self.lambda_cls   * loss_cls
                    + self.lambda_theta * loss_theta)

        # Scale each term by its own loss weight.
        grad_raw[..., 1:5] *= self.lambda_loc
        grad_raw[..., 5] *= self.lambda_theta
        grad_raw[..., 6:] *= self.lambda_cls

        loss_dict = dict(
            total=total_loss, loc=loss_loc,
            obj=loss_obj, noobj=loss_noobj,
            cls=loss_cls, theta=loss_theta
        )
        return total_loss, grad_raw, loss_dict


In [ ]:
# Non-max suppression for overlapping predicted ellipses.
def ellipse_nms(boxes, scores, iou_thresh=0.4, n_samples=200):
    """
    Non-maximum suppression for ellipse detections.
    boxes  : (N, 5) — cx, cy, a, b, theta
    scores : (N,)
    Returns keep indices.
    """
    order = scores.argsort()[::-1]
    keep  = []
    while len(order):
        i = order[0]
        keep.append(i)
        if len(order) == 1:
            break
        rest = order[1:]
        ious = np.array([ellipse_iou_mc(boxes[i], boxes[j], n_samples) for j in rest])
        order = rest[ious < iou_thresh]
    return keep


In [ ]:
# YOLO-style backbone plus fully connected detection head.
class ConvBNLeaky:
    """Conv2D + BatchNorm2D + LeakyReLU — the standard YOLO block."""
    def __init__(self, in_ch, out_ch, kernel=3, stride=1, pad=None):
        self.conv = Conv2D(in_ch, out_ch, kernel, stride, pad)
        self.bn   = BatchNorm2D(out_ch)
        self.act  = LeakyReLU(0.1)

    def forward(self, x, training=True):
        x = self.conv.forward(x, training)
        x = self.bn.forward(x, training)
        x = self.act.forward(x, training)
        return x

    def backward(self, grad):
        grad = self.act.backward(grad)
        grad = self.bn.backward(grad)
        grad = self.conv.backward(grad)
        return grad

    def params(self):
        return self.conv.params() + self.bn.params()

class Conv1x1BNLeaky:
    """Pointwise conv + BatchNorm + LeakyReLU."""
    def __init__(self, in_ch, out_ch):
        self.conv = Conv1x1(in_ch, out_ch)
        self.bn   = BatchNorm2D(out_ch)
        self.act  = LeakyReLU(0.1)

    def forward(self, x, training=True):
        x = self.conv.forward(x, training)
        x = self.bn.forward(x, training)
        x = self.act.forward(x, training)
        return x

    def backward(self, grad):
        grad = self.act.backward(grad)
        grad = self.bn.backward(grad)
        grad = self.conv.backward(grad)
        return grad

    def params(self):
        return self.conv.params() + self.bn.params()

class EllipseNet:
    """
    Full EllipseNet detector.

    Parameters
    ──────────
    img_size  : input spatial size (assumed square after preprocessing)
    S         : grid size (output feature map will be S×S)
    B         : anchors per cell
    C         : number of classes
    anchors   : (B, 2) array of anchor (semi_a, semi_b) in normalised coords
    """

    def __init__(self, img_size=448, S=7, B=2, C=20, anchors=None):
        self.img_size = img_size
        self.S = S
        self.B = B
        self.C = C
        self.out_ch = B * (6 + C)   # per-cell convolutional equivalent
        self.flat_dim = 512 * S * S
        self.fc_hidden = 1024
        self.output_dim = S * S * self.out_ch

        if anchors is None:
            self.anchors = np.array([[0.10, 0.07], [0.20, 0.14]])
        else:
            self.anchors = np.array(anchors)

        # Compact YOLOv1-style backbone. Channels are reduced from the
        # original YOLOv1 so the NumPy-only implementation remains usable.
        self.stem   = ConvBNLeaky(3,   32, 7, stride=2, pad=3)  # 448 -> 224
        self.pool1  = MaxPool2D(2, 2)                          # 224 -> 112
        self.conv2  = ConvBNLeaky(32,  64, 3)
        self.pool2  = MaxPool2D(2, 2)                          # 112 -> 56
        self.pw1    = Conv1x1BNLeaky(64,  32)
        self.conv3  = ConvBNLeaky(32, 128, 3)
        self.pw2    = Conv1x1BNLeaky(128, 64)
        self.conv4  = ConvBNLeaky(64, 128, 3)
        self.pool3  = MaxPool2D(2, 2)                          # 56 -> 28
        self.pw3    = Conv1x1BNLeaky(128, 64)
        self.conv5  = ConvBNLeaky(64, 256, 3)
        self.pw4    = Conv1x1BNLeaky(256, 64)
        self.conv6  = ConvBNLeaky(64, 256, 3)
        self.pool4  = MaxPool2D(2, 2)                          # 28 -> 14
        self.pw5    = Conv1x1BNLeaky(256, 128)
        self.conv7  = ConvBNLeaky(128, 512, 3)
        self.pw6    = Conv1x1BNLeaky(512, 128)
        self.conv8  = ConvBNLeaky(128, 512, 3)
        self.conv9  = ConvBNLeaky(512, 512, 3, stride=2, pad=1) # 14 -> 7
        self.conv10 = ConvBNLeaky(512, 512, 3)
        self.conv11 = ConvBNLeaky(512, 512, 3)
        # YOLOv1-style fully connected detection head, reduced to 1024 hidden units.
        self.fc1    = Dense(self.flat_dim, self.fc_hidden)
        self.fc_act = LeakyReLU(0.1)
        self.fc2    = Dense(self.fc_hidden, self.output_dim)

        self._feature_layers = [
            self.stem, self.pool1,
            self.conv2, self.pool2,
            self.pw1, self.conv3, self.pw2, self.conv4, self.pool3,
            self.pw3, self.conv5, self.pw4, self.conv6, self.pool4,
            self.pw5, self.conv7, self.pw6, self.conv8,
            self.conv9, self.conv10, self.conv11,
        ]
        self._head_layers = [
            self.fc1, self.fc_act, self.fc2,
        ]

    # ─── forward ──────────────────────────────────────────────────────
    def forward(self, x, training=True):
        """
        x : (N, 3, H, W)
        Returns raw output (N, S, S, B, 6+C)
        """
        for layer in self._feature_layers:
            x = layer.forward(x, training)
        # x: (N, 512, S, S) -> FC head -> (N, S, S, B, 6+C)
        N, _, H, W = x.shape
        if H != self.S or W != self.S:
            raise ValueError(
                f"Model output grid is {H}x{W}, but S={self.S}. "
                f"For the YOLO-style setup use img_size=448 and S=7."
            )
        self._feature_shape = x.shape if training else None
        x = x.reshape(N, -1)
        x = self.fc1.forward(x, training)
        x = self.fc_act.forward(x, training)
        x = self.fc2.forward(x, training)
        x = x.reshape(N, self.S, self.S, self.B, 6 + self.C)
        return x

    # ─── backward ─────────────────────────────────────────────────────
    def backward(self, grad_out):
        """
        grad_out : (N, S, S, B, 6+C)
        """
        N = grad_out.shape[0]
        grad = grad_out.reshape(N, self.output_dim)
        grad = self.fc2.backward(grad)
        grad = self.fc_act.backward(grad)
        grad = self.fc1.backward(grad)
        grad = grad.reshape(self._feature_shape)
        for layer in reversed(self._feature_layers):
            grad = layer.backward(grad)
        return grad

    # ─── parameter list ───────────────────────────────────────────────
    def params(self):
        p = []
        for blk in self._feature_layers + self._head_layers:
            if hasattr(blk, 'params'):
                p.extend(blk.params())
        return p

    # ─── inference: decode + NMS ──────────────────────────────────────
    def predict(self, x, conf_thresh=0.8, nms_thresh=0.4, n_samples=200):
        """
        Run full inference on a batch.
        Returns list (len=N) of dicts with 'boxes', 'scores', 'classes'.
        """
        raw = self.forward(x, training=False)
        N = x.shape[0]
        results = []
        S = self.S

        grid_x = np.arange(S).reshape(1, S, 1)
        grid_y = np.arange(S).reshape(S, 1, 1)
        empty = {
            'boxes': np.zeros((0, 5), dtype=np.float32),
            'scores': np.zeros(0, dtype=np.float32),
            'classes': np.zeros(0, dtype=int),
        }

        for ni in range(N):
            r = raw[ni]
            conf = sigmoid(r[..., 0])
            cx = (sigmoid(r[..., 1]) + grid_x) / S
            cy = (sigmoid(r[..., 2]) + grid_y) / S

            # Early checkpoints can produce extreme raw size logits. Clip before
            # exp so invalid ellipses cannot crash ellipse NMS.
            a = np.exp(np.clip(r[..., 3], -6.0, 2.0)) * self.anchors[:, 0]
            b = np.exp(np.clip(r[..., 4], -6.0, 2.0)) * self.anchors[:, 1]
            a = np.clip(a, 1e-4, 0.5)
            b = np.clip(b, 1e-4, 0.5)

            theta = np.pi * sigmoid(r[..., 5])
            cls_p = softmax(r[..., 6:])
            cls_id = cls_p.argmax(axis=-1)
            cls_sc = cls_p.max(axis=-1)
            score = conf * cls_sc

            valid = (
                np.isfinite(score) & np.isfinite(cx) & np.isfinite(cy) &
                np.isfinite(a) & np.isfinite(b) & np.isfinite(theta)
            )
            mask = (score > conf_thresh) & valid

            if not mask.any():
                results.append({k: v.copy() for k, v in empty.items()})
                continue

            boxes = np.stack([cx[mask], cy[mask], a[mask], b[mask], theta[mask]], axis=1)
            scores = score[mask]
            classes = cls_id[mask]

            keep = ellipse_nms(boxes, scores, nms_thresh, n_samples)
            results.append({
                'boxes': boxes[keep],
                'scores': scores[keep],
                'classes': classes[keep],
            })
        return results

    # ─── save / load weights ──────────────────────────────────────────
    def save(self, path):
        """Save all parameters as a .npz archive."""
        arrays = {}
        for i, (p, _) in enumerate(self.params()):
            arrays[f'param_{i}'] = p
        np.savez(path, **arrays)
        print(f"Model saved → {path}")

    def load(self, path):
        """Load parameters from .npz archive."""
        data = np.load(path)
        params = self.params()
        for i, (p, _) in enumerate(params):
            key = f'param_{i}'
            if key in data:
                p[:] = data[key]
        print(f"Model loaded ← {path}")


In [ ]:
# Instantiate model and run a shape sanity check before training.
model = EllipseNet(img_size=IMG_SIZE, S=S, B=B, C=C)
loss_fn = EllipseLoss(S=S, B=B, C=C)

RUN_SANITY_CHECK = True
if RUN_SANITY_CHECK:
    x = np.random.randn(1, 3, IMG_SIZE, IMG_SIZE).astype(np.float32)
    y = model.forward(x, training=True)
    print("Input shape:", x.shape)
    print("Output shape:", y.shape)

print("Expected model output:", (1, S, S, B, 6 + C))
print("Parameter tensors:", len(model.params()))
print("Total scalar parameters:", sum(p.size for p, _ in model.params()))


In [ ]:
# Training and validation loops.
def train_one_epoch(model, loss_fn, optimizer, dataset, batch_size=8,
                    shuffle=True, clip_grad=5.0):
    """
    Runs one training epoch.
    Returns dict of mean losses.
    """
    indices = np.arange(len(dataset))
    if shuffle:
        np.random.shuffle(indices)

    n_batches = max(1, len(indices) // batch_size)
    epoch_losses = {k: 0.0 for k in ['total','loc','obj','noobj','cls','theta']}

    for step in range(n_batches):
        batch_idx = indices[step * batch_size:(step + 1) * batch_size]
        batch = [dataset[i] for i in batch_idx]

        imgs    = np.stack([b[0] for b in batch], axis=0)   # (B,3,H,W)
        targets = [b[1] for b in batch]

        # ── forward ──────────────────────────────────────────────────
        raw  = model.forward(imgs, training=True)
        loss, grad_raw, loss_dict = loss_fn.compute(raw, targets)

        # ── gradient clipping ─────────────────────────────────────────
        grad_norm = np.sqrt((grad_raw**2).sum())
        if grad_norm > clip_grad:
            grad_raw = grad_raw * (clip_grad / (grad_norm + 1e-8))

        # ── backward ──────────────────────────────────────────────────
        optimizer.zero_grad()
        model.backward(grad_raw)
        optimizer.step()

        for k in epoch_losses:
            epoch_losses[k] += loss_dict.get(k, 0.0)

    for k in epoch_losses:
        epoch_losses[k] /= n_batches

    return epoch_losses

def validate(model, loss_fn, dataset, batch_size=8):
    """Run validation without weight updates."""
    indices = np.arange(len(dataset))
    n_batches = max(1, len(indices) // batch_size)
    val_losses = {k: 0.0 for k in ['total','loc','obj','noobj','cls','theta']}

    for step in range(n_batches):
        batch_idx = indices[step * batch_size:(step + 1) * batch_size]
        batch = [dataset[i] for i in batch_idx]
        imgs    = np.stack([b[0] for b in batch], axis=0)
        targets = [b[1] for b in batch]
        raw  = model.forward(imgs, training=False)
        _, _, loss_dict = loss_fn.compute(raw, targets)
        for k in val_losses:
            val_losses[k] += loss_dict.get(k, 0.0)

    for k in val_losses:
        val_losses[k] /= n_batches
    return val_losses

def train(model, loss_fn, train_dataset, val_dataset=None,
          epochs=30, batch_size=8, lr=1e-3,
          checkpoint_dir='checkpoints_allcls', log_path='outputs/train_log_allcls.json',
          patience=10, warmup_epochs=15):
    """
    Full training loop with:
    - Adam optimiser (from scratch)
    - LR warmup then cosine decay
    - Loss curve logging
    - Early stopping
    - Checkpoint saving (best val loss)
    """
    os.makedirs(checkpoint_dir, exist_ok=True)
    os.makedirs(os.path.dirname(log_path) or '.', exist_ok=True)

    optimizer  = Adam(model.params(), lr=lr, weight_decay=1e-4)
    total_steps = epochs * max(1, len(train_dataset) // batch_size)
    step        = 0

    history = []
    best_val_loss = float('inf')
    epochs_no_improve = 0

    print(f"{'Epoch':>6} {'Train':>9} {'Val':>9} {'loc':>7} {'obj':>7} "
          f"{'noobj':>7} {'cls':>7} {'θ':>7}  {'LR':>9}  {'Time':>6}")
    print("─" * 85)

    for epoch in range(1, epochs + 1):
        t0 = time.time()

        # ── learning rate schedule ────────────────────────────────────
        warmup_steps = warmup_epochs * max(1, len(train_dataset) // batch_size)
        new_lr = warmup_cosine_lr(step, warmup_steps, total_steps, lr)
        optimizer.set_lr(new_lr)

        # ── train ─────────────────────────────────────────────────────
        tr = train_one_epoch(model, loss_fn, optimizer, train_dataset, batch_size)
        step += max(1, len(train_dataset) // batch_size)

        # ── validate ──────────────────────────────────────────────────
        vl = None
        if val_dataset is not None:
            vl = validate(model, loss_fn, val_dataset, batch_size)
            val_total = vl['total']
        else:
            val_total = tr['total']

        elapsed = time.time() - t0
        val_str = f"{val_total:9.4f}" if vl else "       -"
        print(f"{epoch:>6} {tr['total']:9.4f} {val_str} "
              f"{tr['loc']:7.4f} {tr['obj']:7.4f} {tr['noobj']:7.4f} "
              f"{tr['cls']:7.4f} {tr['theta']:7.4f}  {new_lr:9.6f}  {elapsed:6.1f}s")

        # ── log ───────────────────────────────────────────────────────
        record = {'epoch': epoch, 'lr': float(new_lr),
                  'train': {k: float(v) for k,v in tr.items()}}
        if vl:
            record['val'] = {k: float(v) for k,v in vl.items()}
        history.append(record)

        # ── checkpoint ────────────────────────────────────────────────
        if val_total < best_val_loss:
            best_val_loss = val_total
            epochs_no_improve = 0
            model.save(f"{checkpoint_dir}/best_model.npz")
        else:
            epochs_no_improve += 1

        # save latest every 5 epochs
        if epoch % 5 == 0:
            model.save(f"{checkpoint_dir}/epoch_{epoch:03d}.npz")

        if epochs_no_improve >= patience:
            print(f"\nEarly stopping at epoch {epoch} (no improvement for {patience} epochs).")
            break

    with open(log_path, 'w') as f:
        json.dump(history, f, indent=2)
    print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")
    print(f"Log saved → {log_path}")
    return history


In [ ]:
# Full training run.
EPOCHS = 20
BATCH_SIZE = 4
LEARNING_RATE = 1e-4
CHECKPOINT_DIR = "checkpoints_allcls"
LOG_PATH = "outputs/train_log_allcls.json"

history = train(
    model,
    loss_fn,
    train_ds,
    val_ds,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    checkpoint_dir=CHECKPOINT_DIR,
    log_path=LOG_PATH,
    patience=EPOCHS,
    warmup_epochs=3,
)


In [ ]:
# Plot a few validation predictions after training.
def show_predictions(model, dataset, class_names, n_images=6, conf_thresh=0.8, out_path="figures/sample_predictions.png"):
    n_images = min(n_images, len(dataset))
    cols = 3
    rows = int(np.ceil(n_images / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for ax, idx in zip(axes, range(n_images)):
        img, targets = dataset[idx]
        display_img = np.clip(img.transpose(1, 2, 0), 0, 1)
        ax.imshow(display_img)
        h, w = display_img.shape[:2]

        pred = model.predict(img[None, ...], conf_thresh=conf_thresh)[0]
        for ellipse, score, cls_id in zip(pred["boxes"], pred["scores"], pred["classes"]):
            cx, cy, a, b, theta = ellipse
            patch = MplEllipse(
                (cx * w, cy * h),
                2 * a * w,
                2 * b * h,
                angle=np.rad2deg(theta),
                fill=False,
                edgecolor="red",
                linewidth=1.6,
            )
            ax.add_patch(patch)
            ax.text(cx * w, cy * h, f"{class_names[int(cls_id)]} {score:.2f}", color="white", fontsize=8)

    fig.tight_layout()
    fig.savefig(out_path, dpi=140)
    plt.show()
    return out_path

if os.path.exists(f"{CHECKPOINT_DIR}/best_model.npz"):
    model.load(f"{CHECKPOINT_DIR}/best_model.npz")
show_predictions(model, val_ds, CLASS_NAMES, conf_thresh=0.8)
